# H&M GCN seed51 · 개인 GPU B · M4 / M5_A / M5_B
개인 계정 azsabae@gmail.com에서 별도 GPU 런타임으로 실행합니다. A/B/C는 서로 다른 모델만 실행하며, 동일 파트를 중복 실행하지 않습니다.

고정300epoch·마지막1주 test·신규상품 추천·전체 카탈로그·uniform 음성1개·하나의 optimizer를 유지합니다. M1~M4 사전학습 없이 각 결합모형도 독립 초기화합니다.
GCN 2층/ID64/batch131072/Adam lr0.0005, 원래 정규화·손실·M2~M4 설계는 그대로입니다. 검증된 COO→CSR 저장형식과 고정 입력 캐시만 사용하며 학습 표현은 매 배치 재계산합니다.

매 epoch 전체 모델·optimizer·난수를 개인 Drive에 업로드·해시 검증합니다. 이 파트 전용 폴더에서만 최신+직전 2개를 남깁니다. 완료 결과는 재사용하고 중단 모델은 마지막 저장 다음 epoch로 재개합니다. 결과와 원본 입력은 삭제하지 않습니다.
3개 런타임 합계 체크포인트 약18GB와 업로드 여유가 필요합니다. 개인 Drive 여유를 먼저 확인하세요.
각 파트 완료는 전체8모형 완료가 아닙니다. 3개 ZIP을 합쳐야 seed51 전체 결과가 됩니다. 1시드만으로 유의성을 주장하지 않습니다.


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import subprocess, sys, json, hashlib, time, uuid, shutil
import torch
assert torch.cuda.is_available(), 'L4 GPU 런타임으로 설정하세요.'
# GPU hardware is recorded; mathematical settings are fixed.
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
drive.mount('/content/drive')
auth.authenticate_user()  # 두 승인 모두 개인계정 선택
import google.auth
from googleapiclient.discovery import build
credentials, _ = google.auth.default()
drive_api = build('drive','v3',credentials=credentials,cache_discovery=False)
owner = drive_api.about().get(fields='user(emailAddress)').execute()['user']['emailAddress']
assert owner == 'azsabae@gmail.com', f'개인 Drive가 아닙니다: {owner}'
SOURCE_COMMIT = '2d8ac73e30e04f779e0d220846610717b24be94e'
REPO = Path('/content/clv-hm-gcn-part-b-s51-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout',SOURCE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip() == SOURCE_COMMIT
sys.path.insert(0,str(REPO))
import clv_hm_gcn_final8_seed51 as screen
import clv_personal_gcn51_checkpoint as storage

# 업로드 제한 시 검증된 이전 저장본을 유지하고 단계적으로 재시도합니다.
exec(compile('"""Retry transient Drive upload limits while retaining checkpoint safeguards."""\nimport json\nimport time\n\n\ndef install(storage, *, sleep=time.sleep):\n    if getattr(storage, \'_upload_retry_installed\', False):\n        return\n    factory = storage.backend\n\n    def backend():\n        module = factory()\n        publish = module.publish\n\n        def publish_with_retry(api, parent, path, ident, epoch):\n            for attempt in range(9):\n                try:\n                    # Each new call checks for an already uploaded matching epoch.\n                    return publish(api, parent, path, ident, epoch)\n                except Exception as exc:\n                    status = getattr(getattr(exc, \'resp\', None), \'status\', None)\n                    try:\n                        errors = json.loads(exc.content)[\'error\'][\'errors\']\n                        reasons = {error.get(\'reason\') for error in errors}\n                    except (AttributeError, KeyError, TypeError, ValueError):\n                        reasons = set()\n                    limited = status == 429 or (status == 403 and bool(\n                        reasons & {\'rateLimitExceeded\', \'userRateLimitExceeded\'}))\n                    if not limited or attempt == 8:\n                        raise\n                    delay = min(300, 60 * 2 ** attempt)\n                    print(f\'Drive 요청 제한: {delay}초 후 재확인 ({attempt + 1}/8). \'\n                          \'로컬과 기존 체크포인트를 보존합니다.\', flush=True)\n                    sleep(delay)\n\n        module.publish = publish_with_retry\n        return module\n\n    storage.backend = backend\n    storage._upload_retry_installed = True\n', '<gcn-drive-retry>', 'exec'), globals())
install(storage)
assert Path(screen.__file__).resolve().parent == REPO.resolve(), '이 런타임에 이전 코드가 남아 있습니다.'
print('개인 Drive 여유 GB:', storage.backend().check_space(drive_api,initial=True)/1e9)
DATA = Path('/content/drive/MyDrive/논문/data')
OUT = DATA / 'results_v3_hm_gcn_seed51_part_b_personal_v1'
OUT.mkdir(parents=True,exist_ok=True)
# 기존 입력 원본의 소유자와 바이트 해시로 두 연결이 같은 계정인지 확인한다.
p = DATA / 'raw/hm/articles.csv'
assert p.is_file(), f'입력 원본 없음: {p}'
h = hashlib.md5()
with p.open('rb') as f:
    for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
        h.update(block)
meta = drive_api.files().get(fileId='1wf9deu3byBxY6Kk3M3v4syK6TWK8hWeY', fields='id,name,md5Checksum,owners(emailAddress,me),trashed').execute()
assert not meta.get('trashed') and meta.get('md5Checksum') == h.hexdigest() and any(o.get('me') and o.get('emailAddress') == owner for o in meta.get('owners', [])), '마운트와 API의 개인 Drive 일치 확인 실패'
print('개인 Drive 원본과 계정 일치 확인 완료')
print('GPU:', torch.cuda.get_device_name(0), '| GCN seed51 | 출력:', OUT)

PART = 'b'
LABELS = ('M4', 'M5_A', 'M5_B')
print('이번 GPU 담당:', LABELS)


## 1. 입력 준비 — 학습 없음
개인 Drive의 원본 두 파일을 바이트 그대로 Colab 로컬 디스크에 복사합니다.
기존 참조 결과 폴더가 있으면 그대로 읽고, 없으면 아래 이름의 ZIP을 references 폴더에서 사용합니다.
ZIP도 없으면 **작은 결과 ZIP 두 개만** 업로드하도록 요청합니다. 원본 데이터 재변환·재학습은 하지 않습니다.

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024**2),b''): h.update(block)
    return h.hexdigest()
EXPECTED = {
 'transactions_train.parquet':'0d90884fc1bc400161e6aeba238cba94183998bedfeb4ebbf4962e17d0c8489b',
 'articles.csv':'25f21882053517f7684ea43f524225a9481a7bb9aab9cf0abfc7fe8c4576a72b'}
LOCAL = Path('/content/hm_gcn_seed51_inputs')
LOCAL.mkdir(exist_ok=True)
for key,name in [('tx_path','transactions_train.parquet'),('item_meta_path','articles.csv')]:
    source=DATA/'raw/hm'/name
    target=LOCAL/name
    if not target.is_file() or sha256(target)!=EXPECTED[name]:
        assert source.is_file(), f'기존 개인 Drive 원본 파일 없음: {source}'
        temporary=target.with_suffix(target.suffix+'.partial')
        shutil.copyfile(source,temporary)
        assert sha256(temporary)==EXPECTED[name], f'원본 바이트 불일치: {name}'
        temporary.replace(target)
    screen.base.v3.SCHEMA['hm'][key]=str(target)
defaults=screen.configure()
ref_specs=[
 (Path(defaults.m2_reference),'clv_m2_adaptive_hm_seed49_lastweek_results.zip'),
 (Path(defaults.m4_reference),'clv_hm_m4c_lastweek_seed49_results.zip')]
refs=[]
for directory,name in ref_specs:
    archive=DATA/'references'/name
    refs.append(directory if directory.is_dir() else archive)
missing=[p for p in refs if not p.exists()]
if missing:
    from google.colab import files
    print('다음 기존 결과 ZIP만 업로드하세요:', [p.name for p in missing])
    uploaded=files.upload()
    for p in missing:
        assert p.name in uploaded, f'누락: {p.name}'
        p.parent.mkdir(parents=True,exist_ok=True)
        p.write_bytes(uploaded[p.name])
cfg=screen.configure(out_dir=str(OUT),m2_reference=str(refs[0]),m4_reference=str(refs[1]))
assert cfg.seeds==(51,) and cfg.epochs==300

# 진단 통계의 미세한 연산 오차만 허용; 고정 protocol과 입력·특징 해시 보존.
import types
_protocol_fix_source = '"""Allow diagnostic reduction roundoff only; preserve frozen protocol bytes."""\nimport ast\nfrom dataclasses import asdict\nimport hashlib\nimport inspect\nimport json\nimport math\nfrom pathlib import Path\n\nLIMITS = {\'item_v_std\': 1e-12,\n          \'user_n_reliability_mean\': 2.5e-7,\n          \'user_v_reliability_mean\': 2.5e-7}\n\n\ndef differences(saved, current):\n    old, new = json.loads(json.dumps(saved)), json.loads(json.dumps(current))\n    a, b = old.pop(\'feature_diagnostics\'), new.pop(\'feature_diagnostics\')\n    if old != new or set(a) != set(b):\n        raise RuntimeError(\'고정 protocol의 입력/설정/특징/진단 항목이 다릅니다\')\n    result = {}\n    for key in a:\n        if a[key] == b[key]:\n            continue\n        x, y = a[key], b[key]\n        if (key not in LIMITS or type(x) not in (int, float)\n                or type(y) not in (int, float) or not math.isfinite(x)\n                or not math.isfinite(y) or abs(x-y) > LIMITS[key]):\n            raise RuntimeError(\'허용하지 않은 feature 진단 차이: \' + key)\n        result[key] = {\'saved\': x, \'current\': y, \'atol\': LIMITS[key]}\n    return result\n\n\ndef guard(path, current, atomic_json):\n    path = Path(path)\n    if not path.exists():\n        atomic_json(path, current)\n        return current\n    original = path.read_bytes()\n    saved = json.loads(original)\n    delta = differences(saved, current)\n    if delta:\n        digest = hashlib.sha256(json.dumps(current, sort_keys=True).encode()).hexdigest()\n        audit = path.with_name(\'diagnostic_roundoff_\' + digest[:16] + \'.json\')\n        atomic_json(audit, {\'protocol_sha256\': hashlib.sha256(original).hexdigest(),\n                           \'differences\': delta, \'current_feature_diagnostics\': current[\'feature_diagnostics\'],\n                           \'unchanged\': \'all other protocol fields including exact feature/input hashes and initialization\'})\n        print(\'[protocol 확인] 입력·특징·설정 동일; 진단 소수점 차이만 기록, 기존 원본 보존\', flush=True)\n    assert path.read_bytes() == original\n    return saved\n\n\nOLD = \'\'\'    if protocol_path.exists() and json.loads(protocol_path.read_text()) != json.loads(json.dumps(protocol)):\n        raise RuntimeError("기존 실행 경로의 고정 protocol이 다릅니다")\n    io._atomic_json(protocol_path, protocol)\'\'\'\nNEW = \'\'\'    protocol = _protocol_roundoff_guard(protocol_path, protocol, io._atomic_json)\'\'\'\n\n\ndef patched_source(module):\n    source = inspect.getsource(getattr(module, \'_before_roundoff_prepare\', module.prepare))\n    if source.count(OLD) != 1:\n        raise RuntimeError(\'고정 prepare 소스가 예상과 다릅니다; 수정하지 않습니다\')\n    return source.replace(OLD, NEW)\n\n\ndef install(module):\n    if getattr(module.prepare, \'_roundoff_guard_installed\', False):\n        return\n    scope = dict(module.__dict__, _protocol_roundoff_guard=guard)\n    source = patched_source(module)\n    exec(compile(source, \'<m2-protocol-roundoff>\', \'exec\'), scope)\n    module._before_roundoff_prepare = module.prepare\n    module.prepare = scope[\'prepare\']\n    module.prepare._roundoff_guard_installed = True\n\n\ndef recover_frame(module, frame):\n    """Finish only the already-computed input stage; no graph reallocation."""\n    if (frame.f_code.co_name != \'prepare\'\n            or Path(frame.f_code.co_filename).resolve() != Path(module.__file__).resolve()):\n        raise RuntimeError(\'입력 준비 오류 프레임 신원 불일치\')\n    values = dict(frame.f_locals)\n    module._validate_config(values[\'cfg\'])\n    assert module._feature_hash(values[\'features\']) == values[\'feature_sha\']\n    differences(json.loads(values[\'protocol_path\'].read_text()), values[\'protocol\'])\n    function = ast.parse(patched_source(module)).body[0]\n    start = next(i for i, node in enumerate(function.body) if isinstance(node, ast.Assign)\n                 and any(isinstance(t, ast.Name) and t.id == \'protocol_path\' for t in node.targets))\n    body = function.body[start:]\n    assert isinstance(body[-1], ast.Return)\n    body[-1] = ast.Assign(targets=[ast.Name(id=\'_recovered\', ctx=ast.Store())], value=body[-1].value)\n    scope = dict(module.__dict__, **values, _protocol_roundoff_guard=guard)\n    exec(compile(ast.fix_missing_locations(ast.Module(body=body, type_ignores=[])),\n                 \'<finish-verified-m2-inputs>\', \'exec\'), scope)\n    recovered = scope[\'_recovered\']\n    assert recovered[\'data\'] is values[\'data\']\n    return recovered, asdict(values[\'cfg\'])\n\n\ndef compare_gcn_protocol(saved, current, compare_graph, graph_cfg):\n    """Ignore diagnostics only after the runner\'s original graph gates pass."""\n    old, new = json.loads(json.dumps(saved)), json.loads(json.dumps(current))\n    old_graph, new_graph = old.pop(\'m3_audit\'), new.pop(\'m3_audit\')\n    old_cmp, new_cmp = old.pop(\'m3_reference_comparison\'), new.pop(\'m3_reference_comparison\')\n    for record, graph in ((old_cmp, old_graph), (new_cmp, new_graph)):\n        expected = compare_graph(record[\'reference\'], graph, graph_cfg)\n        if record != json.loads(json.dumps(expected)):\n            raise RuntimeError(\'M3 진단/참조 비교 기록의 내부 불일치\')\n    if old_cmp[\'reference\'] != new_cmp[\'reference\'] or old_cmp[\'rule\'] != new_cmp[\'rule\']:\n        raise RuntimeError(\'M3 고정 참조/검증 규칙 변경\')\n    comparison = compare_graph(old_graph, new_graph, graph_cfg)\n    x = old[\'m4_weight_audit\'].pop(\'train_mean_weight\')\n    y = new[\'m4_weight_audit\'].pop(\'train_mean_weight\')\n    if not all(math.isfinite(v) for v in (x,y)) or abs(x-y) > 1e-12:\n        raise RuntimeError(\'M4 가중치 평균 차이 허용 범위 초과\')\n    # This includes the exact M4 weight SHA256, source file hashes and all settings.\n    if old != new:\n        raise RuntimeError(\'GCN 실제 입력·설정·특징·가중치·소스가 달라 재개하지 않습니다\')\n    return {\'m3_comparison\': comparison, \'m4_train_mean\': [x,y]}\n\n\ndef bind_existing_gcn_run(screen, prepared, expected_run_hash):\n    path = Path(prepared[\'cfg\'].out_dir) / expected_run_hash / \'protocol.json\'\n    original = path.read_bytes()\n    saved = json.loads(original)\n    assert hashlib.sha256(json.dumps(saved, sort_keys=True).encode()).hexdigest()[:12] == expected_run_hash\n    graph_cfg = screen.base.hm.configure(out_dir=str(Path(prepared[\'cfg\'].out_dir)/\'inputs\'))\n    audit = compare_gcn_protocol(saved, prepared[\'protocol\'], screen.compare_m3_audit, graph_cfg)\n    current = prepared[\'protocol\']\n    digest = hashlib.sha256(json.dumps(current, sort_keys=True).encode()).hexdigest()\n    screen.base.io._atomic_json(path.with_name(\'resume_diagnostics_\'+digest[:16]+\'.json\'),\n        {\'saved_protocol_sha256\': hashlib.sha256(original).hexdigest(),\n         \'current_protocol\': current, \'compatibility\': audit,\n         \'rule\': \'reuse original graph gates; exact non-diagnostic fields and weight hashes\'})\n    assert path.read_bytes() == original\n    prepared.update(protocol=saved, config_hash=expected_run_hash, run_dir=path.parent)\n    print(\'[기존 실행 연결] 입력·특징·가중치·설정·소스 동일; 원래 protocol/체크포인트 경로 보존:\', expected_run_hash, flush=True)\n    return prepared\n\n\ndef install_gcn(screen, expected_run_hash):\n    if getattr(screen.prepare, \'_protocol_resume_hash\', None) == expected_run_hash:\n        return\n    original = screen.prepare\n    def prepare(cfg):\n        return bind_existing_gcn_run(screen, original(cfg), expected_run_hash)\n    prepare._protocol_resume_hash = expected_run_hash\n    screen.prepare = prepare\n'
assert hashlib.sha256(_protocol_fix_source.encode()).hexdigest() == '350a349aa64f4924880e6a2ca7c9a314b5e73bb18fc1cc84c0fbaa60c5a59f7f'
protocol_fix = types.ModuleType('clv_m2_protocol_roundoff')
exec(compile(_protocol_fix_source, '<verified-protocol-roundoff>', 'exec'), protocol_fix.__dict__)
protocol_fix.install(screen.base.m2)
protocol_fix.install_gcn(screen, '73149b37a2fb')
# Reuse a verified preparation in this runtime: rebuilding it duplicates large GPU graphs.
if 'prepared' in globals():
    from dataclasses import asdict
    assert asdict(prepared['cfg']) == asdict(cfg), '기존 준비 설정이 다릅니다. 저장 상태 확인 후 새 런타임을 사용하세요.'
    assert prepared['revision'] == SOURCE_COMMIT and prepared['input_hash'] == '50f1fc88d1144b2f2e739d4956536560ccc6fe444ba3fd10c55276d8cbdf762c', '기존 준비의 코드 또는 입력이 다릅니다.'
    assert all(k in prepared for k in ('gcn_binary_adj','gcn_m3_adj','references','config_hash','run_dir'))
    assert getattr(screen.SafeProgressStore, '_personal_storage_installed', False), '기존 저장 설정을 확인하세요.'
    checkpoint_record = json.loads((prepared['run_dir']/'personal_checkpoint_storage.json').read_text())
    assert checkpoint_record['account'] == 'azsabae@gmail.com'
    checkpoint_folder = checkpoint_record['folder_id']
    print('기존 준비 데이터·저장 설정 재사용: GPU 그래프를 다시 만들지 않습니다.', flush=True)
else:
    prepared=screen.prepare(cfg)
    checkpoint_folder=storage.install(screen,drive_api,prepared)
assert set(prepared['data']['splits'])=={'test'}
assert prepared['data'].get('loss_w') is None
print('사전점검 완료:', LABELS, '/ seed51 / 300epoch')
print('결과 폴더:', prepared['run_dir'])
print('체크포인트 폴더:', 'https://drive.google.com/drive/folders/'+checkpoint_folder)


## 2. 담당 모델만 순차 학습: M4, M5_A, M5_B
매 epoch 원격 저장 완료 후 다음 epoch로 갑니다. 다른 파트 모델은 실행하지 않습니다.


In [ ]:
assert getattr(screen.SafeProgressStore,'_personal_storage_installed',False)
assert cfg.seeds == (51,) and LABELS == ('M4', 'M5_A', 'M5_B')
try:
    for label in LABELS:
        screen.run_arm(prepared, label)
        screen.report(prepared)
        print('담당 모델 완료:', label, flush=True)
    assert all(screen.completed(prepared, label) is not None for label in LABELS)
    print('PART COMPLETE:', PART, LABELS)
finally:
    output=screen.export_zip(prepared,'/content/hm_gcn_seed51_part_b_results.zip')
    backup=prepared['run_dir']/Path(output).name
    shutil.copyfile(output,backup)
    assert sha256(output)==sha256(backup)
    print('현재까지 결과 ZIP:', backup)


## 3. 담당 모델 저장 결과 — 추가 학습·평가 없음


In [ ]:
from IPython.display import display
result=screen.report(prepared)
display(result['absolute'])
print('이 GPU 담당:', LABELS, '| 전체8개는 A/B/C 결과를 합쳐 확인합니다.')


## 4. 결과 다운로드
ZIP은 성능·설정·진단·학습이력만 포함합니다. 대용량 재개 체크포인트는 별도 개인 Drive 폴더에 유지합니다.
다운로드가 멈추면 출력된 Drive 결과 폴더에서 ZIP을 직접 받으세요.

In [ ]:
from google.colab import files
output=screen.export_zip(prepared,'/content/hm_gcn_seed51_part_b_results.zip')
backup=prepared['run_dir']/Path(output).name
shutil.copyfile(output,backup)
assert sha256(output)==sha256(backup)
print('Drive ZIP:',backup)
files.download(output)
